<a href="https://colab.research.google.com/github/omarsamehabobaker619-bot/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarsamehabobaker619-bot/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**ML task type: Scoring (Ranking)**

My lane is CTR / Engagement Opportunity Scoring. The decision it supports is:
which pages should the content team look at first?

This isn't classification (sorting pages into fixed groups) or clustering
(grouping similar pages with no labels). Each page gets a score for how much its
CTR seems to underperform — that's the scoring part. But the score only really
matters in relation to other pages, since the output is an ordered shortlist for
review — that's the ranking part. So this task is really both: scoring to produce
the number, ranking to turn it into a usable list.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** how far a page's CTR falls below what's typical for pages in a
similar position.

There's no column in the data that says "this page needs a rewrite," so this is
a proxy — a stand-in for something I can't measure directly (this idea comes from
Google's ML problem-framing guide). My rough plan: compare each page's CTR to the
typical CTR of pages ranked similarly, and use that gap as the target.

This proxy isn't perfect. A page could have low CTR for reasons that have nothing
to do with the content itself — search intent, seasonality, or a bad snippet
that's easy to fix but different from the deeper issue this score is meant to
catch. I'm keeping this in mind rather than treating the score as ground truth.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@K**

The output is a short list of pages to review, not a judgment on all 30,000
pages. So what matters is whether the pages at the *top* of that list are
actually worth reviewing — not overall accuracy across everything. Precision@K
(e.g. top 50) checks exactly that: of the pages flagged, how many are real,
meaningful gaps rather than noise.

I'll also compare against a simple rule (like "flag anything under a fixed CTR
cutoff") to see if scoring actually does better than just picking a number and
sticking to it.

When a page shows up near the top of this list, the action is concrete: a content
editor reviews it and updates the title, meta description, or on-page content to
better match what searchers are looking for.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df_clean = df.replace([np.inf, -np.inf], np.nan)

lane_cols = ["impressions_90d", "avg_position", "ctr"]
lane_slice = df_clean[lane_cols].dropna()

print("Unit of analysis: one row = one page")
print("Rows:", len(lane_slice))
lane_slice.head()

Unit of analysis: one row = one page
Rows: 30000


,impressions_90d,avg_position,ctr
0,3803,10.6,0.76
1,15320,20.3,0.05
2,12581,36.5,0.09
3,11751,6.2,0.49
4,19140,44.0,0.13


In [ ]:
# Sketch: how far each page's CTR is from the overall typical CTR
overall_median_ctr = lane_slice["ctr"].median()
lane_slice["target_ctr_gap"] = lane_slice["ctr"] - overall_median_ctr

print("Overall median CTR:", overall_median_ctr)
print("\nSketch of the target column (a few example rows):")
lane_slice[["avg_position", "ctr", "target_ctr_gap"]].head(10)

Overall median CTR: 0.07

Sketch of the target column (a few example rows):


,avg_position,ctr,target_ctr_gap
0,10.6,0.76,0.69
1,20.3,0.05,-0.02
2,36.5,0.09,0.02
3,6.2,0.49,0.42
4,44.0,0.13,0.06
5,8.5,0.03,-0.04
6,7.0,0.00,-0.07
7,21.2,0.06,-0.01
8,46.0,0.09,0.02
9,4.9,0.16,0.09


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why not just a fixed rule?**

A simple rule like "flag anything under 5% CTR" treats every page the same,
no matter its position. But a page ranked #1 and a page ranked #8 don't have
the same "normal" CTR to begin with — a flat cutoff would either flag too many
fine pages or miss real problems buried lower down.

Scoring pages relative to similar pages (instead of one number for everyone)
gives a fairer picture of which pages are actually underperforming for their
situation, not just which ones have a low raw number.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- [x] Names the ML task type, target/proxy, and success metric
- [x] Shows the unit of analysis as a real dataframe
- [x] Explains why this is ML, not just a fixed rule
- [x] Ties the output to a real content action (rewrite titles/meta/content)
- [x] Notebook runs top to bottom with no errors (Restart session → Run all)
- [x] Committed to work/notebooks/ in my repo